In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch03-bias-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["numpy==1.26.4", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "03-06-public-data-bias.py"

# 公開資料的來源分布

同一文獻可貢獻許多列；這份有界查詢不是隨機樣本，不能估計未發表失敗率。預設使用內嵌的官方回應；REFRESH_LIVE 可重新擷取，數字可能改變。練習加在第三格末尾，再執行第四格。


In [ ]:
%%writefile "{script}"
from pathlib import Path
import json
import hashlib
import urllib.request
from datetime import datetime, timezone
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Genuine official API responses, frozen for standalone reproduction.
# Canonical JSON bytes are verified before parsing. No missing label is filled.
SNAPSHOT_JSON = '{"pages":[{"activities":[{"action_type":null,"activity_comment":null,"activity_id":32260,"activity_properties":[],"assay_chembl_id":"CHEMBL674637","assay_description":"Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"Cc1cc(C)c(/C=C2\\\\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)c32)[nH]1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":{"bei":"19.25","le":"0.37","lle":"2.94","sei":"8.93"},"modality":null,"molecule_chembl_id":"CHEMBL68920","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL68920","pchembl_value":"7.39","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119482,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"41.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.041"},{"action_type":null,"activity_comment":null,"activity_id":32263,"activity_properties":[],"assay_chembl_id":"CHEMBL621151","assay_description":"Inhibition of autophosphorylation of human epidermal growth factor-receptor (EGF-R) expressed in A431 cells","assay_type":"F","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000219","bao_label":"cell-based format","canonical_smiles":"Cc1cc(C)c(/C=C2\\\\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)c32)[nH]1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL68920","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL68920","pchembl_value":"6.52","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119482,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"300.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.3"},{"action_type":null,"activity_comment":null,"activity_id":32265,"activity_properties":[],"assay_chembl_id":"CHEMBL615325","assay_description":"Inhibition of ligand-induced proliferation in human EGF-R Kinase expressing 3T3 cells","assay_type":"F","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000219","bao_label":"cell-based format","canonical_smiles":"Cc1cc(C)c(/C=C2\\\\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)c32)[nH]1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL68920","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL68920","pchembl_value":"5.11","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119482,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"7820.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"7.82"},{"action_type":null,"activity_comment":null,"activity_id":32267,"activity_properties":[],"assay_chembl_id":"CHEMBL674637","assay_description":"Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\\\\C(=O)Nc2ncnc(Nc3ccc(F)c(Cl)c3)c21","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":{"bei":"14.02","le":"0.27","lle":"3.16","sei":"6.03"},"modality":null,"molecule_chembl_id":"CHEMBL69960","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL69960","pchembl_value":"6.77","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119494,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"170.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.17"},{"action_type":null,"activity_comment":null,"activity_id":32270,"activity_properties":[],"assay_chembl_id":"CHEMBL621151","assay_description":"Inhibition of autophosphorylation of human epidermal growth factor-receptor (EGF-R) expressed in A431 cells","assay_type":"F","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000219","bao_label":"cell-based format","canonical_smiles":"Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\\\\C(=O)Nc2ncnc(Nc3ccc(F)c(Cl)c3)c21","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL69960","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL69960","pchembl_value":"7.40","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119494,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"40.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.04"},{"action_type":null,"activity_comment":null,"activity_id":32272,"activity_properties":[],"assay_chembl_id":"CHEMBL615325","assay_description":"Inhibition of ligand-induced proliferation in human EGF-R Kinase expressing 3T3 cells","assay_type":"F","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000219","bao_label":"cell-based format","canonical_smiles":"Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\\\\C(=O)Nc2ncnc(Nc3ccc(F)c(Cl)c3)c21","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL69960","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL69960","pchembl_value":"6.36","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119494,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"440.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.44"},{"action_type":null,"activity_comment":null,"activity_id":32680,"activity_properties":[],"assay_chembl_id":"CHEMBL677833","assay_description":"In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (human carcinoma).","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1145114","document_journal":"J Med Chem","document_year":2003,"ligand_efficiency":{"bei":"13.62","le":"0.25","lle":"0.26","sei":"6.40"},"modality":null,"molecule_chembl_id":"CHEMBL137635","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL137635","pchembl_value":"5.03","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":262238,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"9300.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"9.3"},{"action_type":null,"activity_comment":null,"activity_id":32770,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)O)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL306988","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL306988","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137177,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"500000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"500.0"},{"action_type":null,"activity_comment":null,"activity_id":32772,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"O=C(O)/C=C/c1ccc(O)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL66879","molecule_pref_name":"COUMARIC ACID","parent_molecule_chembl_id":"CHEMBL66879","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137215,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"3000000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"3000.0"},{"action_type":null,"activity_comment":null,"activity_id":32780,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=Cc1cc(O)ccc1[N+](=O)[O-]","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":{"bei":"18.67","le":"0.34","lle":"2.29","sei":"3.62"},"modality":null,"molecule_chembl_id":"CHEMBL77085","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL77085","pchembl_value":"4.02","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137197,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"96000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"96.0"},{"action_type":null,"activity_comment":null,"activity_id":33406,"activity_properties":[],"assay_chembl_id":"CHEMBL674637","assay_description":"Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"Cc1cc(C(=O)NCCN2CCOCC2)[nH]c1/C=C1\\\\C(=O)N(C)c2ncnc(Nc3ccc(F)c(Cl)c3)c21","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":{"bei":"9.77","le":"0.19","lle":"2.04","sei":"4.57"},"modality":null,"molecule_chembl_id":"CHEMBL443268","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL443268","pchembl_value":"5.28","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119495,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"5310.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"5.31"},{"action_type":null,"activity_comment":null,"activity_id":34039,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"COc1cc(/C=C(\\\\C#N)C(=O)O)cc(OC)c1O","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL76979","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL76979","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137172,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"264000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"264.0"},{"action_type":null,"activity_comment":null,"activity_id":34041,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=C(N)/C(C#N)=C/c1ccc(O)cc1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":{"bei":"29.22","le":"0.52","lle":"5.34","sei":"5.87"},"modality":null,"molecule_chembl_id":"CHEMBL76589","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL76589","pchembl_value":"6.90","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137232,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"125.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.125"},{"action_type":null,"activity_comment":null,"activity_id":34045,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"COc1cc(C=C(C#N)C#N)cc(OC)c1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL77724","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL77724","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137189,"relation":">","src_id":1,"standard_flag":1,"standard_relation":">","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"1250000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"1250.0"},{"action_type":null,"activity_comment":null,"activity_id":34047,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=Cc1ccc(N2CCOCC2)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL420385","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL420385","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137185,"relation":">","src_id":1,"standard_flag":1,"standard_relation":">","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"625000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"625.0"},{"action_type":null,"activity_comment":null,"activity_id":34049,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=Cc1ccc(O)c(O)c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":{"bei":"23.93","le":"0.43","lle":"2.93","sei":"5.06"},"modality":null,"molecule_chembl_id":"CHEMBL76904","molecule_pref_name":"TYRPHOSTIN 23","parent_molecule_chembl_id":"CHEMBL76904","pchembl_value":"4.46","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137194,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"35000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"35.0"},{"action_type":null,"activity_comment":null,"activity_id":34051,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N[C@@H](Cc1ccc(O)c(O)c1)C(=O)O","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL1009","molecule_pref_name":"LEVODOPA","parent_molecule_chembl_id":"CHEMBL1009","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137202,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"900000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"900.0"},{"action_type":null,"activity_comment":null,"activity_id":34669,"activity_properties":[],"assay_chembl_id":"CHEMBL674637","assay_description":"Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"CCN(CC)CC(O)CNC(=O)c1cc(C)c(/C=C2\\\\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)c32)[nH]1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1134862","document_journal":"Bioorg Med Chem Lett","document_year":2002,"ligand_efficiency":{"bei":"17.24","le":"0.34","lle":"5.78","sei":"6.91"},"modality":null,"molecule_chembl_id":"CHEMBL304271","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL304271","pchembl_value":"9.35","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":119504,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"0.45","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.00045"},{"action_type":null,"activity_comment":null,"activity_id":35294,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#C/C(=C\\\\c1ccc(O)c(O)c1)C(N)=O","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":{"bei":"24.49","le":"0.46","lle":"4.51","sei":"4.66"},"modality":null,"molecule_chembl_id":"CHEMBL296407","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL296407","pchembl_value":"5.00","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137234,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"10000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"10.0"},{"action_type":null,"activity_comment":null,"activity_id":35296,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=C(N)/C(C#N)=C/c1ccc(O)c([N+](=O)[O-])c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":{"bei":"15.01","le":"0.27","lle":"2.75","sei":"2.63"},"modality":null,"molecule_chembl_id":"CHEMBL309625","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL309625","pchembl_value":"4.22","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137231,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"60000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"60.0"}],"page_meta":{"limit":20,"next":"/chembl/api/data/activity.json?limit=20&offset=20&target_chembl_id=CHEMBL203&standard_type=IC50&order_by=activity_id","offset":0,"previous":null,"total_count":26600},"url":"https://www.ebi.ac.uk/chembl/api/data/activity.json?target_chembl_id=CHEMBL203&standard_type=IC50&limit=20&order_by=activity_id"},{"activities":[{"action_type":null,"activity_comment":null,"activity_id":35297,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"COc1ccc(NN=C(C#N)C#N)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL77825","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL77825","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137225,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"600000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"600.0"},{"action_type":null,"activity_comment":null,"activity_id":35301,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"COc1cc(C=C(C#N)C#N)cc(OC)c1O","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL421877","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL421877","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137187,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"850000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"850.0"},{"action_type":null,"activity_comment":null,"activity_id":36518,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=Cc1cc(O)c(O)c(O)c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":{"bei":"27.32","le":"0.50","lle":"4.29","sei":"5.10"},"modality":null,"molecule_chembl_id":"CHEMBL310798","molecule_pref_name":"TYRPHOSTIN 25","parent_molecule_chembl_id":"CHEMBL310798","pchembl_value":"5.52","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137192,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"3000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"3.0"},{"action_type":null,"activity_comment":null,"activity_id":37510,"activity_properties":[],"assay_chembl_id":"CHEMBL677833","assay_description":"In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (human carcinoma).","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"C/N=N/Nc1ccc2ncnc(N(C)c3ccccc3)c2c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1145114","document_journal":"J Med Chem","document_year":2003,"ligand_efficiency":{"bei":"17.43","le":"0.32","lle":"1.29","sei":"7.75"},"modality":null,"molecule_chembl_id":"CHEMBL135592","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL135592","pchembl_value":"5.10","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":262235,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"8000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"8.0"},{"action_type":null,"activity_comment":null,"activity_id":40025,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#C/C(=C\\\\c1ccc([N+](=O)[O-])cc1)C(=O)O","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL309866","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL309866","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137171,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"1300000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"1300.0"},{"action_type":null,"activity_comment":null,"activity_id":40026,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#C/C(=C/c1ccc[nH]1)C(=O)O","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL77381","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL77381","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137219,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"1360000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"1360.0"},{"action_type":null,"activity_comment":null,"activity_id":41220,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#C/C(=C\\\\c1ccc(O)cc1)C(=O)O","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL76602","molecule_pref_name":"alpha-CYANO-4-HYDROXYCINNAMIC ACID","parent_molecule_chembl_id":"CHEMBL76602","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137236,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"165000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"165.0"},{"action_type":null,"activity_comment":null,"activity_id":41222,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=Cc1ccncc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL77737","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL77737","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137217,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"250000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"250.0"},{"action_type":null,"activity_comment":null,"activity_id":41224,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=Cc1ccc(F)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL76983","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL76983","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137188,"relation":">","src_id":1,"standard_flag":1,"standard_relation":">","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"1250000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"1250.0"},{"action_type":null,"activity_comment":null,"activity_id":41226,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=Cc1ccc(O)c([N+](=O)[O-])c1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL308134","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL308134","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137198,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"125000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"125.0"},{"action_type":null,"activity_comment":null,"activity_id":42300,"activity_properties":[],"assay_chembl_id":"CHEMBL677833","assay_description":"In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (human carcinoma).","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"CN(C)/N=N/c1ccc2ncnc(Nc3cccc(Cl)c3)c2c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1145114","document_journal":"J Med Chem","document_year":2003,"ligand_efficiency":{"bei":"24.03","le":"0.47","lle":"3.26","sei":"11.94"},"modality":null,"molecule_chembl_id":"CHEMBL136491","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL136491","pchembl_value":"7.85","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":262243,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"14.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.014"},{"action_type":null,"activity_comment":null,"activity_id":42432,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"CC(=C(C#N)C#N)c1ccc(N)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL77280","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL77280","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137181,"relation":">","src_id":1,"standard_flag":1,"standard_relation":">","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"1250000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"1250.0"},{"action_type":null,"activity_comment":null,"activity_id":42433,"activity_properties":[],"assay_chembl_id":"CHEMBL674645","assay_description":"Inhibition of EGF-dependent epidermal growth factor receptor autophosphorylation (inactive)","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"O=[N+]([O-])c1ccc(O)c(O)c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL42423","molecule_pref_name":"4-Nitrocatechol","parent_molecule_chembl_id":"CHEMBL42423","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137226,"relation":null,"src_id":1,"standard_flag":0,"standard_relation":null,"standard_text_value":"Not active","standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":null,"target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":"Not Active","toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":null},{"action_type":null,"activity_comment":null,"activity_id":43662,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CCNC(=O)/C(C#N)=C/c1ccc(O)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL76557","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL76557","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137178,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"625000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"625.0"},{"action_type":null,"activity_comment":null,"activity_id":44685,"activity_properties":[],"assay_chembl_id":"CHEMBL677833","assay_description":"In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (human carcinoma).","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"Cc1cccc(Nc2ncnc3ccc(N)cc23)c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1145114","document_journal":"J Med Chem","document_year":2003,"ligand_efficiency":{"bei":"23.97","le":"0.43","lle":"2.74","sei":"9.40"},"modality":null,"molecule_chembl_id":"CHEMBL336113","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL336113","pchembl_value":"6.00","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":262227,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"1000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"1.0"},{"action_type":null,"activity_comment":null,"activity_id":44690,"activity_properties":[],"assay_chembl_id":"CHEMBL677833","assay_description":"In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (human carcinoma).","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"C/N=N/Nc1ccc2ncnc(Nc3cccc(Cl)c3)c2c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1145114","document_journal":"J Med Chem","document_year":2003,"ligand_efficiency":{"bei":"22.38","le":"0.43","lle":"2.56","sei":"9.39"},"modality":null,"molecule_chembl_id":"CHEMBL133024","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL133024","pchembl_value":"7.00","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":262245,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"100.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"0.1"},{"action_type":null,"activity_comment":null,"activity_id":44695,"activity_properties":[],"assay_chembl_id":"CHEMBL677833","assay_description":"In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (human carcinoma).","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"CN(C)/N=N/c1ccc2ncnc(N(C)c3ccccc3)c2c1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1145114","document_journal":"J Med Chem","document_year":2003,"ligand_efficiency":{"bei":"15.91","le":"0.29","lle":"0.91","sei":"8.55"},"modality":null,"molecule_chembl_id":"CHEMBL344652","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL344652","pchembl_value":"4.88","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":262234,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"13350.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"13.35"},{"action_type":null,"activity_comment":null,"activity_id":44885,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"N#CC(C#N)=CNc1ccc(O)cc1","data_validity_comment":"Outside typical range","data_validity_description":"Values for this activity type are unusually large/small, so may not be accurate","document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL77814","molecule_pref_name":null,"parent_molecule_chembl_id":"CHEMBL77814","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137223,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"350000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"350.0"},{"action_type":null,"activity_comment":null,"activity_id":44886,"activity_properties":[],"assay_chembl_id":"CHEMBL674643","assay_description":"Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor receptor kinase","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"O=CN/C=C/c1cc(O)ccc1O","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":{"bei":"27.09","le":"0.51","lle":"4.04","sei":"6.98"},"modality":null,"molecule_chembl_id":"CHEMBL47986","molecule_pref_name":"ERBSTATIN","parent_molecule_chembl_id":"CHEMBL47986","pchembl_value":"4.85","potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137221,"relation":"=","src_id":1,"standard_flag":1,"standard_relation":"=","standard_text_value":null,"standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":"14000.0","target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":null,"toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":"14.0"},{"action_type":null,"activity_comment":null,"activity_id":44964,"activity_properties":[],"assay_chembl_id":"CHEMBL674646","assay_description":"Inhibition of EGF-dependent epidermal growth factor receptor autophosphorylation (inactive)","assay_type":"B","assay_variant_accession":null,"assay_variant_mutation":null,"bao_endpoint":"BAO_0000190","bao_format":"BAO_0000357","bao_label":"single protein format","canonical_smiles":"O=C(O)c1ccc(O)cc1","data_validity_comment":null,"data_validity_description":null,"document_chembl_id":"CHEMBL1124610","document_journal":"J Med Chem","document_year":1989,"ligand_efficiency":null,"modality":null,"molecule_chembl_id":"CHEMBL441343","molecule_pref_name":"PARABEN","parent_molecule_chembl_id":"CHEMBL441343","pchembl_value":null,"potential_duplicate":0,"qudt_units":"http://www.openphacts.org/units/Nanomolar","record_id":137240,"relation":null,"src_id":1,"standard_flag":0,"standard_relation":null,"standard_text_value":"Not active","standard_type":"IC50","standard_units":"nM","standard_upper_value":null,"standard_value":null,"target_chembl_id":"CHEMBL203","target_organism":"Homo sapiens","target_pref_name":"Epidermal growth factor receptor","target_tax_id":"9606","text_value":"Not Active","toid":null,"type":"IC50","units":"uM","uo_units":"UO_0000065","upper_value":null,"value":null}],"page_meta":{"limit":20,"next":"/chembl/api/data/activity.json?limit=20&offset=40&target_chembl_id=CHEMBL203&standard_type=IC50&order_by=activity_id","offset":20,"previous":"/chembl/api/data/activity.json?limit=20&offset=0&target_chembl_id=CHEMBL203&standard_type=IC50&order_by=activity_id","total_count":26600},"url":"https://www.ebi.ac.uk/chembl/api/data/activity.json?limit=20&offset=20&target_chembl_id=CHEMBL203&standard_type=IC50&order_by=activity_id"}],"retrieved_utc":"2026-11-04T07:53:51.871842+00:00","selection":"First 40 EGFR IC50 activities ordered by activity_id, two pages of 20; not representative.","status":{"activities":24527044,"chembl_db_version":"ChEMBL_37","chembl_release_date":"2026-05-01","compound_records":3824604,"disinct_compounds":2921148,"publications":101100,"status":"UP","targets":18552}}'
EXPECTED_SHA256 = 'bcd9b6dae678a73394e1051d95eb7017e35070ab4ddcf832b6104ed0cca728e3'
assert hashlib.sha256(SNAPSHOT_JSON.encode()).hexdigest() == EXPECTED_SHA256
snapshot = json.loads(SNAPSHOT_JSON)
REFRESH_LIVE = False
if REFRESH_LIVE:
    pages = []
    for page in snapshot["pages"]:
        request = urllib.request.Request(page["url"], headers={"Accept": "application/json"})
        with urllib.request.urlopen(request, timeout=60) as response:
            payload = json.load(response)
        pages.append({"url": page["url"], "page_meta": payload["page_meta"], "activities": payload["activities"]})
    snapshot["pages"] = pages
    snapshot["retrieved_utc"] = datetime.now(timezone.utc).isoformat()
    with urllib.request.urlopen("https://www.ebi.ac.uk/chembl/api/data/status.json", timeout=60) as response:
        snapshot["status"] = json.load(response)
    Path("03-06-live-snapshot.json").write_text(json.dumps(snapshot, ensure_ascii=False, indent=2))
records = [row for page in snapshot["pages"] for row in page["activities"]]
df = pd.DataFrame(records)
assert len(df) == 40 and df.activity_id.is_unique
assert df.target_chembl_id.eq("CHEMBL203").all()
assert df.standard_type.eq("IC50").all()
assert df.activity_id.is_monotonic_increasing
summary = df.groupby("document_chembl_id").agg(
    activity_rows=("activity_id", "size"),
    molecules=("molecule_chembl_id", "nunique"),
    assays=("assay_chembl_id", "nunique"),
).sort_values("activity_rows", ascending=False)
summary["row_share"] = summary["activity_rows"] / len(df)
print("Retrieved UTC:", snapshot["retrieved_utc"])
print("Frozen snapshot SHA256:", EXPECTED_SHA256)
print("Data mode:", "LIVE (may differ from book)" if REFRESH_LIVE else "FROZEN")
for page in snapshot["pages"]:
    print("Query:", page["url"])
print(summary.to_string())
print("Distinct molecules:", df.molecule_chembl_id.nunique())
print("Distinct assays:", df.assay_chembl_id.nunique())
relation = df.standard_relation.map({"=": "Exact (=)", ">": "Lower bound (>)"}).fillna("Other / missing")
counts = pd.crosstab(df.document_chembl_id, relation).reindex(
    index=summary.index, columns=["Exact (=)", "Lower bound (>)", "Other / missing"], fill_value=0)
assert counts.to_numpy().sum() == len(df)
fig, ax = plt.subplots(figsize=(10, 4.8))
counts.plot.barh(stacked=True, ax=ax, color=["#296780", "#CB8D36", "#B5B9BF"], width=.6)
ax.invert_yaxis()
ax.set(xlabel="Activity records (40-record ordered prefix)", ylabel="Source document",
       title="EGFR IC50 records: source concentration")
ax.set_xlim(0, summary.activity_rows.max()*1.28)
for y, (_, row) in enumerate(summary.iterrows()):
    ax.text(row.activity_rows+.35, y, f"{int(row.activity_rows)} ({row.row_share:.0%})", va="center")
ax.legend(title="Reported relation", loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(.02,.01,"Ordered by activity_id, not a random sample. No active/inactive threshold applied.",fontsize=9)
fig.tight_layout(rect=(0,.04,1,1))
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "03-06-public-data-bias.png", dpi=180)
plt.close(fig)


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "03-06-public-data-bias.png")))